# Qwen3.5-9B · Run All 자동 다운로드·학습·추론

**기존 베이스라인과 같은 폴더에 이 파일을 넣고 기존 CUDA 환경의 커널에서 Run All을 실행하세요.**

1. 데이터 경로·CUDA 확인 및 필요한 패키지 설치
2. `Qwen/Qwen3.5-9B` 다운로드 — 이미 완성된 로컬 모델은 재사용
3. 4bit LoRA 학습 → 검증 loss → 전체 test 추론 → 제출 CSV 저장

준비할 것은 인터넷 연결과 기존 `data/train.csv`, `data/test.csv`, `data/train/`, `data/test/`입니다.
최초 모델 다운로드는 약 19.3GB이며 추가 패키지·체크포인트를 위한 디스크 여유가 필요합니다. 패키지와 기본 모델 다운로드만 네트워크를 사용하고 학습·추론은 Python 로컬 모델로 수행합니다.

- 권장 대상: RTX 5060 Ti **16GB**. NF4 4bit, batch=1, gradient checkpointing, 384² 픽셀 예산. 실제 GPU 실행·VRAM·성능은 아직 미검증입니다.
- 유지: 원본 23셀, seed42, 200→180/20 분할, LoRA r8/alpha16/dropout.05, lr1e-4, epoch1, 전체 토큰 loss, 기존 프롬프트·파서·greedy2토큰.
- 추가 설정: non-thinking, 생성 부분 디코딩, BF16(미지원 시 FP16), 동결 임베딩/출력층 저정밀, 단일 GPU SDPA.
- 모델 경로: `downloads/models/Qwen3.5-9B/`.
- 결과: `output/TASK-002/<실행시각>/submission.csv`, `qwen3_5_9b_lora/`, `run_config.json`.
- 기존 2.5 모델·어댑터·제출 파일은 덮어쓰지 않습니다. Run All을 다시 하면 완료된 모델 파일을 재사용하고 새 학습 실행을 만듭니다.

새로 연 커널에서 실행하면 설치 후 다음 셀로 이어집니다. 이전 실행에서 이미 불러온 패키지가 교체되면 재시작 안내와 함께 중단합니다. 이 경우 **Restart Kernel → Run All**로 다시 실행하세요. 다운로드가 중단된 경우에도 다시 실행하면 Hub 캐시를 이용해 이어받기를 시도합니다.

TASK-002 / EXP-004 / v1.2. 실제 전체 실행은 사용자 환경에서 확인 필요.

# 자동 환경 준비

설치되지 않았거나 버전 조건을 충족하지 못하는 패키지만 자동 설치합니다. 기존 CUDA PyTorch는 보존합니다.
PyTorch가 없는 환경에서는 원본 기준 CUDA 12.8 빌드를 설치합니다. GPU 드라이버 설치·업데이트는 자동화하지 않습니다.
패키지 설치 실패·CUDA 사용 불가·데이터 누락 시 오류를 표시하고 모델 다운로드/학습으로 넘어가지 않습니다.

In [1]:
import os, sys, json, subprocess, importlib, importlib.metadata as metadata
from pathlib import Path

print("Run All: 환경 준비 → 모델 다운로드 → 학습 → 추론 → 제출 파일 생성")
# 이후 다운로드는 새 Python 프로세스에서 실행하여, 이전 커널의 HF 오프라인 상수 캐시와 분리합니다.

Run All: 환경 준비 → 모델 다운로드 → 학습 → 추론 → 제출 파일 생성


In [2]:
ASSET_DIR = "downloads"
LIB_DIR = os.path.join(ASSET_DIR, "libs")
MODEL_REPO_ID = "Qwen/Qwen3.5-9B"
MODEL_DIR = os.path.join(ASSET_DIR, "models", "Qwen3.5-9B")
DATA_DIR = "data"
from datetime import datetime
OUTPUT_DIR = os.path.join("output", "TASK-002", datetime.now().strftime("%Y%m%d_%H%M%S_%f"))

for filename in ["train.csv", "test.csv"]:
    if not os.path.isfile(os.path.join(DATA_DIR, filename)):
        raise FileNotFoundError(f"{DATA_DIR}/{filename}가 없습니다. 기존 베이스라인 폴더에서 실행하세요.")
for folder in ["train", "test"]:
    if not os.path.isdir(os.path.join(DATA_DIR, folder)):
        raise FileNotFoundError(f"{DATA_DIR}/{folder}/ 이미지 폴더가 없습니다.")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print("모델:", MODEL_REPO_ID)
print("결과 폴더:", os.path.abspath(OUTPUT_DIR))

모델: Qwen/Qwen3.5-9B
결과 폴더: c:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-002\20260921_114505_175306


In [3]:
# 기존 CUDA PyTorch를 유지. 없을 때만 원본 CUDA 빌드를 설치합니다.
try:
    torch_version = metadata.version("torch")
    print("기존 PyTorch 유지:", torch_version)
except metadata.PackageNotFoundError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--index-url", "https://download.pytorch.org/whl/cu128",
        "torch==2.11.0+cu128", "torchvision==0.26.0+cu128"
    ], check=True)
    importlib.invalidate_caches()
    print("CUDA PyTorch 설치 완료")

기존 PyTorch 유지: 2.11.0+cu128


In [4]:
# 패키지를 교체하기 전에 커널에 torch를 import하지 않도록 별도 프로세스에서 점검.
subprocess.run([sys.executable, "-c", """
import torch
assert torch.cuda.is_available(), 'CUDA 사용 불가: 기존 baseline GPU 커널/드라이버를 확인하세요.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')
"""], check=True)

CompletedProcess(args=['c:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\baseline\\Scripts\\python.exe', '-c', "\nimport torch\nassert torch.cuda.is_available(), 'CUDA 사용 불가: 기존 baseline GPU 커널/드라이버를 확인하세요.'\nprint('PyTorch:', torch.__version__)\nprint('GPU:', torch.cuda.get_device_name(0))\nprint('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')\n"], returncode=0)

In [5]:
# 1) 필요한 의존성만 설치. 이미 설치된 CUDA torch 버전은 pip constraint로 고정.
import tempfile
try:
    from packaging.requirements import Requirement
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "packaging"], check=True)
    from packaging.requirements import Requirement

requirements = ["transformers>=5.8.0,<6.0.0", "peft>=0.18.0", "accelerate>=0.34.2",
                "bitsandbytes>=0.43.3", "huggingface_hub", "pandas", "Pillow", "tqdm", "torchvision"]
def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

def missing_requirements(requirements):
    missing = []
    for value in requirements:
        req = Requirement(value)
        version = installed_version(req.name)
        if version is None or (req.specifier and not req.specifier.contains(version, prereleases=True)):
            missing.append(value)
    return missing

needed = missing_requirements(requirements)
if needed:
    watched = {"torch": "torch", "torchvision": "torchvision", "transformers": "transformers",
               "peft": "peft", "accelerate": "accelerate", "bitsandbytes": "bitsandbytes",
               "huggingface_hub": "huggingface_hub", "tokenizers": "tokenizers", "numpy": "numpy",
               "PIL": "Pillow", "pandas": "pandas", "safetensors": "safetensors"}
    loaded_before = {dist: installed_version(dist) for module, dist in watched.items() if module in sys.modules}
    with tempfile.TemporaryDirectory() as temp_dir:
        constraint = Path(temp_dir) / "keep_torch.txt"
        constraint.write_text("torch==" + metadata.version("torch") + "\n", encoding="utf-8")
        print("필요한 패키지 설치:", needed)
        subprocess.run([sys.executable, "-m", "pip", "install", "--constraint", str(constraint), *needed], check=True)
    importlib.invalidate_caches()
    changed_loaded = [dist for dist, old in loaded_before.items() if installed_version(dist) != old]
    if changed_loaded:
        raise RuntimeError(f"이미 불러온 패키지가 변경됐습니다: {changed_loaded}. Restart Kernel → Run All을 한 번 실행하세요.")
if missing_requirements(requirements):
    raise RuntimeError("필요한 패키지 버전이 설치되지 않았습니다. 위 설치 로그를 확인하세요.")

# 2) 완료된 모델은 재사용. 설정/토크나이저/인덱스와 모든 shard의 헤더·길이를 확인합니다.
# 파일 전체의 암호학적 무결성을 검증하는 것은 아닙니다.
def local_model_complete(model_dir):
    folder = Path(model_dir)
    required = ["config.json", "tokenizer_config.json", "tokenizer.json", "preprocessor_config.json",
                "video_preprocessor_config.json", "chat_template.jinja", "model.safetensors.index.json"]
    try:
        if any(not (folder / name).is_file() or (folder / name).stat().st_size == 0 for name in required):
            return False
        for name in required:
            if name.endswith(".json"):
                json.loads((folder / name).read_text(encoding="utf-8"))
        config = json.loads((folder / "config.json").read_text(encoding="utf-8"))
        if config.get("model_type") != "qwen3_5" or config.get("text_config", {}).get("hidden_size") != 4096:
            raise ValueError("MODEL_DIR의 config가 Qwen3.5-9B와 다릅니다. 경로를 확인하세요.")
        weight_map = json.loads((folder / "model.safetensors.index.json").read_text(encoding="utf-8"))["weight_map"]
        if not weight_map:
            return False
        for name in set(weight_map.values()):
            path = folder / name
            if not path.is_file():
                return False
            with path.open("rb") as stream:
                header_length = int.from_bytes(stream.read(8), "little")
                if not 0 < header_length <= min(100_000_000, path.stat().st_size-8):
                    return False
                header = json.loads(stream.read(header_length))
            offsets = [item["data_offsets"][1] for key, item in header.items() if key != "__metadata__"]
            if not offsets or max(offsets) != path.stat().st_size - 8 - header_length:
                return False
            if any(key not in header for key, shard in weight_map.items() if shard == name):
                return False
        return True
    except (OSError, KeyError, TypeError, json.JSONDecodeError, UnicodeDecodeError):
        return False

DOWNLOAD_CODE = r"""
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
repo_id, local_dir = sys.argv[1:3]
folder = Path(local_dir)
record_path = folder / "download_revision.json"
record = json.loads(record_path.read_text(encoding="utf-8")) if record_path.is_file() else {}
revision = record.get("revision") if record.get("repo_id") == repo_id else None
if not revision:
    revision = HfApi().model_info(repo_id).sha
record_path.write_text(json.dumps({"repo_id": repo_id, "revision": revision}, indent=2), encoding="utf-8")
snapshot_download(repo_id=repo_id, revision=revision, local_dir=local_dir,
                  allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja", "LICENSE"], max_workers=4)
print("모델 다운로드 완료:", local_dir)
"""

def ensure_model(model_dir, repo_id, runner=None):
    runner = subprocess.run if runner is None else runner
    Path(model_dir).mkdir(parents=True, exist_ok=True)
    if local_model_complete(model_dir):
        print("다운로드 완료된 모델 재사용:", model_dir)
        return "reused"
    print("Qwen3.5-9B 다운로드 시작/재개 (최초 약 19.3GB). 진행 표시를 기다려주세요.")
    download_env = os.environ.copy()
    download_env["HF_HUB_OFFLINE"] = "0"
    download_env["TRANSFORMERS_OFFLINE"] = "0"
    download_env["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
    runner([sys.executable, "-u", "-c", DOWNLOAD_CODE, repo_id, str(Path(model_dir).resolve())],
           env=download_env, check=True)
    if not local_model_complete(model_dir):
        raise RuntimeError("모델 다운로드가 불완전합니다. 다운로드 오류를 확인하고 다시 Run All하세요.")
    return "downloaded"

try:
    download_status = ensure_model(MODEL_DIR, MODEL_REPO_ID)
except BaseException as exc:
    with open(os.path.join(OUTPUT_DIR, "setup_failure.json"), "w", encoding="utf-8") as f:
        json.dump({"stage": "model_download", "error": str(exc)}, f, ensure_ascii=False, indent=2)
    raise
revision_path = Path(MODEL_DIR) / "download_revision.json"
MODEL_REVISION = json.loads(revision_path.read_text(encoding="utf-8")).get("revision") if revision_path.is_file() else None
# 3) 다운로드 완료 후 학습·추론에서는 로컬 파일만 사용.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("준비 완료. 다음 셀부터 데이터 로드 → 학습 → 추론을 진행합니다.")

필요한 패키지 설치: ['transformers>=5.8.0,<6.0.0']
Qwen3.5-9B 다운로드 시작/재개 (최초 약 19.3GB). 진행 표시를 기다려주세요.
준비 완료. 다음 셀부터 데이터 로드 → 학습 → 추론을 진행합니다.


# 데이터 준비

데이터셋은 사전에 배포되어 `data` 폴더에 아래 구조로 준비되어 있어야 합니다. 다운로드·압축 해제 작업은 없습니다.

- data/train.csv, data/train 폴더
- data/test.csv, data/test 폴더
- data/sample_submission.csv


# 라이브러리, 데이터, 설정

In [6]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen3_5ForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# 자동 다운로드/재사용 확인을 마친 로컬 Qwen3.5-9B 폴더
MODEL_ID = MODEL_DIR
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 2  # 원본 generate 실제 길이 유지
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드 : 사전 배포된 data 폴더
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# 학습데이터 200개만 추출
train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

c:\Users\SSAFY\Desktop\AI2_Challenge\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0921 11:49:46.839000 30728 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Device: cuda


# 모델, Processor

로컬 `downloads/models/Qwen3.5-9B/`를 읽고 새 LoRA를 생성합니다.
Qwen3_5ForConditionalGeneration을 지원하는 Transformers가 필요합니다.
단일 GPU에 올리며 CPU offload로 자동 우회하지 않습니다. 기본 모델이 로드된 뒤 큰 동결 임베딩/출력층의 FP32 확장을 되돌려 메모리를 절약합니다. norm은 PEFT 준비 상태를 유지합니다.

Windows에서 선택적 Gated DeltaNet 고속 커널이 없으면 PyTorch 대체 구현이 사용될 수 있어 속도/메모리 실측이 필요합니다.
[아키텍처·커널 문서](https://huggingface.co/docs/transformers/model_doc/qwen3_5)

In [7]:
# 공식 모델 아키텍처와 로컬 파일 확인
assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."
assert os.path.isfile(os.path.join(MODEL_ID, "config.json")), "앞의 자동 다운로드 셀을 먼저 실행하세요."
print("GPU VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GiB")

# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
    local_files_only=True,   # 로컬 파일만 사용
)

# 사전학습 모델
base_model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa",
    trust_remote_code=True,
    local_files_only=True,   # 로컬 파일만 사용
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
# PEFT가 FP32로 올린 큰 동결 임베딩/출력층만 BF16/FP16으로 복원.
# LoRA와 normalization 파라미터를 일괄 저정밀 변환하지 않음.
for frozen_module in (base_model.get_input_embeddings(), base_model.get_output_embeddings()):
    if frozen_module is not None:
        assert not any(p.requires_grad for p in frozen_module.parameters())
        if type(frozen_module.weight).__name__ != "Params4bit":
            frozen_module.to(dtype=COMPUTE_DTYPE)
torch.cuda.empty_cache()
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False
base_model.config.text_config.use_cache = False  # 학습 중 생성 캐시 비활성화

# LoRA 세팅
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 최소 재현 기록: 모델/패키지/데이터 설정과 실제 LoRA 파라미터 수
import json, hashlib, importlib.metadata as metadata
with open(os.path.join(DATA_DIR, "train.csv"), "rb") as f:
    train_csv_sha256 = hashlib.sha256(f.read()).hexdigest()
with open(os.path.join(MODEL_ID, "config.json"), "rb") as f:
    model_config_sha256 = hashlib.sha256(f.read()).hexdigest()
run_config = {
    "task_id": "TASK-002", "experiment_id": "EXP-004", "model_id": "Qwen/Qwen3.5-9B",
    "model_path": os.path.abspath(MODEL_ID), "model_revision": MODEL_REVISION, "download_status": download_status, "seed": SEED, "sample_rows": 200,
    "train_rows": 180, "valid_rows": 20, "image_pixel_budget": IMAGE_SIZE**2,
    "max_new_tokens": MAX_NEW_TOKENS, "enable_thinking": False,
    "train_csv_sha256": train_csv_sha256, "model_config_sha256": model_config_sha256,
    "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "packages": {p: metadata.version(p) for p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate"]},
    "compute_dtype": str(COMPUTE_DTYPE),
    "frozen_embedding_output_dtype": str(COMPUTE_DTYPE),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_vram_bytes": torch.cuda.get_device_properties(0).total_memory,
    "status": "model_loaded_training_not_started"
}
with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

GPU VRAM: 15.9 GiB


Loading weights: 100%|██████████| 760/760 [00:12<00:00, 58.95it/s] 


trainable params: 14,548,992 || all params: 9,424,362,736 || trainable%: 0.1544


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [8]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [9]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(os.path.join(DATA_DIR, row["path"])).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [10]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# 현재 실행의 분할을 기록. 과거 3B 실행과 동일한지는 train.csv 해시/ID로 대조.
pd.concat([
    train_subset[["id", "path"]].assign(split="train"),
    valid_subset[["id", "path"]].assign(split="valid")
]).to_csv(os.path.join(OUTPUT_DIR, "split_manifest.csv"), index=False)

# fine-tuning

원본과 같이 180행으로 1 epoch 학습합니다. loss 정의·학습률·gradient accumulation은 유지합니다.
모델 크기 증가로 학습 시간과 VRAM은 새로 확인해야 합니다. 기존 3B에서 측정한 시간은 적용할 수 없습니다.

In [11]:
from tqdm.auto import tqdm

# 모델은 로드 때 CUDA:0에 배치됨. 4bit 모델을 다시 이동하지 않음.
GRAD_ACCUM = 4

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 1 * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.amp.GradScaler("cuda", enabled=(COMPUTE_DTYPE == torch.float16))

# 학습 루프
import time
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
training_started = time.perf_counter()
global_step = 0
for epoch in range(1):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        if epoch == 0 and step == 1:
            print("첫 배치 최대 할당 VRAM:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GiB")
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

torch.cuda.synchronize()
run_config["training_and_validation_seconds"] = time.perf_counter() - training_started
run_config["training_peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
run_config["training_peak_reserved_bytes"] = torch.cuda.max_memory_reserved()

# 모델 저장
SAVE_DIR = os.path.join(OUTPUT_DIR, "qwen3_5_9b_lora")
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)

run_config["status"] = "training_completed"
run_config["valid_loss"] = val_loss / val_steps
run_config["adapter_path"] = os.path.abspath(SAVE_DIR)
with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

Epoch 1 [train]:   0%|          | 0/180 [00:00<?, ?batch/s]c:\Users\SSAFY\Desktop\AI2_Challenge\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
Epoch 1 [train]:   1%|          | 1/180 [00:01<04:04,  1.36s/batch]

첫 배치 최대 할당 VRAM: 12.05 GiB


Epoch 1 [valid]: 100%|██████████| 20/20 [00:06<00:00,  3.15batch/s]


[Epoch 1] valid loss 0.4236
Saved: output\TASK-002\20260921_114505_175306\qwen3_5_9b_lora


# inference

학습한 Qwen3.5 LoRA로 전체 test를 추론하고 새 실행 폴더에 제출 CSV를 저장합니다.
non-thinking + greedy 2토큰이며, 입력 길이 이후의 생성 토큰만 디코딩합니다.
원본 파서와 기본값 `a` 정책은 이번 모델 교체에서 그대로 유지합니다. 파서 실패를 숨기는 한계와 생성 Accuracy 부재는 남아 있습니다.
실행 시간·점수는 아직 측정하지 않았습니다.

In [12]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 추론을 위해 모든 레이어 활성화
model.eval()
model.config.use_cache = True
model.config.text_config.use_cache = True
preds = []

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
inference_started = time.perf_counter()

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(os.path.join(DATA_DIR, row["path"])).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad(), torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    # print("output_text:", output_text)
    # print("extract_choice:", extract_choice(output_text))
    preds.append(extract_choice(output_text))

torch.cuda.synchronize()
run_config["inference_seconds"] = time.perf_counter() - inference_started
run_config["inference_peak_allocated_bytes"] = torch.cuda.max_memory_allocated()

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved", SUBMISSION_PATH)
run_config["status"] = "test_inference_completed_submission_not_uploaded"
run_config["submission_path"] = os.path.abspath(SUBMISSION_PATH)
with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

Inference:   0%|          | 0/6714 [00:00<?, ?sample/s]c:\Users\SSAFY\Desktop\AI2_Challenge\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
Inference: 100%|██████████| 6714/6714 [43:00<00:00,  2.60sample/s]

Saved output\TASK-002\20260921_114505_175306\submission.csv


In [13]:
# 모델 응답 예시
print(output_text)

d
